# 09 · 第一个基线：大模型直接猜

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → [09 第一个基线 ✅] → 10 第一个工具 ✅ → 11 起：更多工具、agent …
```

> **这一章的代码在合并请求 #9 里。** 调大模型要花钱，所以只有第 3 步会真的调模型（20 道题，约 1 分钱）；200 道题的评测已经跑过、记在 MLflow 里，下面直接读记录，不会重复花钱。

## 背景：agent 要先打败的对手

计划里有四个对照（第 00 章）。这一章做第一个：**把症状交给大模型，让它一次说出最可能的 10 个病**，不给工具、不查资料。

为什么先做它：
- 它回答"**大模型自己知道多少**"。agent 要证明查资料、用工具有用，就必须比它好
- 它第一次把整条路走通：密钥、调用、格式、记账、追踪、评测，全部用真的模型跑一遍

要先说清一个局限：这些病例来自已经发表的论文，**模型训练时可能读过**。它答对了，可能是推理出来的，也可能是记住的。所以这个分数是"它自己能做到的上限"。以后用模型训练截止之后才发表的病例再考一次，就能看出有多少是记忆。

## 这一章做的 6 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 密钥和接口 | `.env` + OpenAI 标准接口 | 密钥不进代码；换模型只改配置 |
| 2 | 提示词和答卷格式 | JSON 模式 + pydantic 校验 | 答案能被程序读；格式错了有补救 |
| 3 | 先小考 | ci-20，20 道题 | 花 1 分钱先确认整条路是通的 |
| 4 | 每次调用都记下来 | `mlflow.openai.autolog()` | 模型收到什么、答了什么、用了多少 token |
| 5 | 思考模式值不值 | 同一份题，两种模式比 | 用评测做决定，不凭感觉 |
| 6 | 大考和看错题 | dev-200 + 错题分析 | 第一个可信的准确率；知道错在哪 |

**花费**：这一章所有评测加起来约 0.4 美元，其中思考模式那 20 道题就占了 0.25 美元。

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

> 运行 `uv` 的命令时，有时会先出现几行 `Building` / `Installed`：那是 uv 发现代码有变化，顺手更新了一下环境，可以忽略。

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
for d in ("/opt/homebrew/bin", "/usr/local/bin", str(Path.home() / ".local/bin")):  # 编辑器不是从终端打开时，也找得到 uv
    if Path(d).is_dir() and d not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] += os.pathsep + d
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 密钥和接口

### 在干什么
把调用大模型要用的**密钥**放好，并确认能连上模型服务。

### 背景
密钥相当于银行卡密码：谁拿到谁就能用你的余额。最常见的事故，是密钥写进了代码、被推到 GitHub 上。

### 用了什么技术，为什么

| 东西 | 作用 |
|---|---|
| `.env` 文件 | 密钥只放在这里。`.gitignore` 第一行就是它，永远不进 git（第 02 章） |
| `SecretStr` | 程序读进来的密钥被"包起来"，打印、报错时都显示成 `**********` |
| OpenAI 官方接口（SDK） | DeepSeek 和 OpenAI 用同一套接口，所以用最标准的客户端去连 |

**为什么用 OpenAI 的接口连 DeepSeek**：这套接口已经是行业标准，DeepSeek、通义、Kimi，还有医院自己部署的模型（vLLM），都支持它。所以换模型只改两个设置（地址和模型名），**代码一行不动**。这也是"数据不出医院"的前提：把地址换成医院内网的模型就行。

### 公司里什么情况怎么做
- **密钥**：本机放 `.env`；服务器上放在专门的密钥管理服务里（AWS Secrets Manager 这类），不写在任何文件里
- **密钥泄露了**：马上去服务商网站作废它、换一个新的。删掉那次提交不够
- **团队共用**：每人一个自己的密钥，用量能分开看

### 执行
检查密钥和连接，并列出服务上有哪些模型（这一步不花钱）。再看一眼 `.env` 的模板，模板里没有真的密钥：

In [ ]:
run("uv run raredx llm check")
run("cat .env.example")

### 结果怎么看
- `models deepseek-flash, deepseek-v4-pro`：服务上有两个模型
- `using deepseek-flash (available)`：我们用的是便宜、快的 flash
- 模板里 `DEEPSEEK_API_KEY=` 后面是空的；真的密钥只在你电脑上的 `.env` 里。**笔记里永远不打印 `.env`**

### 面试怎么说
> "The model is reached through the OpenAI-compatible SDK, so switching provider — or pointing at a model hosted inside the hospital — is a configuration change. The key lives in a git-ignored .env and is held as a SecretStr."

---

## 第 2 步 · 提示词和答卷格式

### 在干什么
写好给模型的话（**提示词**，prompt），并要求它**按固定格式**交答卷，程序才能自动判分。

### 背景
大模型默认回答一段自由的文字，比如"我觉得最可能是 Perrault 综合征……"。这样的文字没法自动和标准答案比对。

### 用了什么技术，为什么

| 做法 | 作用 |
|---|---|
| 提示词写在代码里 | 改提示词和改代码一样，要走合并请求审查；每次评测记下版本号 `direct-v1` |
| 要求 JSON 格式 | 模型只能回答一个固定结构：`candidates` 里 10 个 `{病名, OMIM 编号, 基因}` |
| JSON 模式 | 服务商提供的开关，保证回答是合法的 JSON |
| pydantic 校验 | 读进来的答卷必须符合规定的格式，不符合当场报错 |
| 补救一次 | 格式不对，把错误告诉模型，让它重答一次；再错就算这道题没答 |

> 提示词里只有症状和性别，**没有任何答案的部分**：有一条测试专门检查这一点（第 07 章说的"评测隔离"）。

### 公司里什么情况怎么做
- **提示词谁来改**：工程师改的，放在代码里、走审查（我们是这种）；产品、医生也要改的，放进专门的"提示词管理"（MLflow 也有，叫 Prompt Registry）
- **改了提示词**：版本号加一，在 dev-200 上重跑，和上一版比

### 执行
看看第 05 章那道 Perrault 的题，模型实际收到的是什么：

In [ ]:
from raredx.baselines.llm_direct import PROMPT_VERSION, SYSTEM_PROMPT, user_prompt
from raredx.data import cases as case_store, sources
from raredx.evals.systems import question_of

source = sources.load_sources(ROOT / "evals/datasets.toml")["phenopacket-store"]
by_id = {c.id: c for c in case_store.read(sources.processed_dir(source, ROOT / "data"))}
case = by_id["PMID_39701103_Proband_F3"]

print(f"提示词版本：{PROMPT_VERSION}\n")
print("—— 系统提示（每道题都一样）——")
print(SYSTEM_PROMPT)
print("\n—— 这道题 ——")
print(user_prompt(question_of(case)))

### 结果怎么看
- 系统提示：说明身份、要 10 个最可能的诊断、每个要病名 + OMIM 编号 + 基因，并给出 JSON 的样子
- 这道题：性别和 5 个症状（每个带标准编号）。**没有病名、没有基因、没有来源论文**

### 面试怎么说
> "The prompt is versioned in code and asks for strict JSON; replies are validated with pydantic, with one repair attempt, and a reply that still fails is recorded and scored as a miss."

---

## 第 3 步 · 先小考：20 道题试跑

### 在干什么
在 ci-20（第 06 章那 20 道题）上跑一次，确认从调用到判分整条路是通的，再花钱跑 200 道。

### 背景
直接跑 200 道，如果哪里有问题（密钥、格式、网络），钱花了、结果却不能用。先用 20 道题，一分钱试错。

> **一个真实踩过的坑：又一次线程锁死。** 第一次跑 200 道题时，程序又卡住了：CPU 为 0，DeepSeek 的回答早就回来了，但没人去读。查下来是 MLflow 的"使用情况统计"（telemetry）：它在每个打分线程里临时导入两个模块，多个线程同时导入时互相锁死，和第 08 章 litellm 那次是同一类问题。办法是**把这个统计功能关掉**（官方提供了开关）。这也正好：它本来会把使用数据发到 MLflow 的服务器，而这个项目不应该往外发任何东西。所有 MLflow 的设置现在集中在 [`src/raredx/mlflow_setup.py`](../../src/raredx/mlflow_setup.py)，每条都写了原因。

### 公司里什么情况怎么做
- **先小后大**：新的模型、新的提示词，都先在小题集上跑通，再上大题集
- **看花费**：每次评测都记下用了多少 token、大概多少钱（这一步会打印出来）

### 执行
这一格会真的调模型，20 道题，约 1 分钱，十秒左右：

In [ ]:
run("uv run raredx eval run --system llm-direct --split ci-20")

### 结果怎么看
- 最后几行是我们的分数表，`20 cases, 20 answered`：20 道题都交了合格的答卷
- 前 10 名命中率在 30%–40% 之间（每次略有不同）。20 道题的误差范围很宽（第 07 章），这里只看"通不通"，不看分数
- `usage:` 一行：用了多少 token、按高峰价大约多少钱、平均每道题几秒

每次运行的分数会有一点不同：大模型的回答本身有随机性，这很正常。

### 面试怎么说
> "New models and prompts run on a 20-case smoke set first, so wiring, format and cost problems surface for a cent rather than for the full run."

---

## 第 4 步 · 每次调用都记进 MLflow

### 在干什么
每一次调大模型，都自动记成一条**追踪记录**里的一层：模型收到的完整提示词、它的回答、用了多少 token、花了多久。

### 背景
第 08 章的追踪记录里只有"题目"和"答卷"两头。现在中间多了一个模型调用，出了问题要能看到：模型到底收到了什么、答了什么。

### 用了什么技术，为什么
**`mlflow.openai.autolog()`**：MLflow 自带的"自动记录"，打开以后，每一次通过 OpenAI 接口的调用都会自动记下来，不用改调用的代码。这是 MLflow 官方推荐的做法：现成的框架用自动记录，自己写的部分才手动加。

一条追踪记录（trace）记的是**一道题从头到尾的经过**。它不是文件夹，更像一张流水账：顶格的一条记"这道题"，缩进的一条记"这道题当中做的一个步骤"。每一条都有开始和结束的时间、收到了什么、交出了什么。

```
llm-direct        2.1 秒   ← 这道题：收到症状 → 交出 10 个候选
  └─ Completions  2.1 秒   ← 其中一步：调用模型（提示词、回答、token）
```

`llm-direct` 是我们起的名字（系统的名字）；`Completions` 是 MLflow 自动起的，来自这次调用的接口名 chat completions，意思是"把对话发给模型，模型接着写出回答"，也就是**问了模型一次**。如果答卷格式不对、补救了一次，同一道题下面就会有两条 `Completions`。

打个比方：实验记录里写"做了一次 PCR，用时 2 小时"，下面缩进写"上机扩增，用时 1.5 小时"。缩进的意思是"这一步是上面那件事里的一部分"。MLflow 里，顶格的一整条叫 **trace**，里面的每一步叫 **span**。

现在一道题只有一步，还看不出这样记的好处。到了 agent，一道题里会有十几步：查术语、查疾病、调好几次模型。哪一步慢、哪一步出错，一眼就能看出来。

### 公司里什么情况怎么做
- **算钱**：线上服务按追踪记录里的 token 数算每个用户、每个功能花了多少
- **排查**：用户说"答错了"，找到那次的追踪记录，直接看模型收到了什么

### 执行
取出刚才那次 20 道题评测里的一条追踪记录：

In [ ]:
from raredx.config import settings
from raredx.evals.harness import Harness
import mlflow

harness = Harness(settings.mlflow_tracking_uri, settings.mlflow_artifact_root)
latest = next(r for r in harness.runs("ci-20") if r["name"] == "llm-direct")
trace = harness.traces(latest["run_id"])[0]

for span in trace.data.spans:
    print(f"{span.name:<12} 类型 {span.span_type:<11} 用时 {(span.end_time_ns - span.start_time_ns) / 1e9:.1f} 秒")
print("\n这道题用的 token：", trace.info.token_usage)
model_call = trace.data.spans[1]
print("\n模型收到的最后一条消息（题目）：")
print(model_call.inputs["messages"][-1]["content"][:300])
print("\n模型回答的开头：")
print(model_call.outputs["choices"][0]["message"]["content"][:200], "…")

### 结果怎么看
- 两层：`llm-direct`（这道题）和它下面的 `Completions`（调用模型），模型那层几乎占了全部用时
- token：输入三百左右（提示词 + 症状），输出几百（10 个候选的 JSON）
- 模型收到的就是第 2 步看到的那段题目；回答是 JSON

在网页上看更直观：运行 `uv run mlflow ui --port 5050`（第 08 章），打开实验 **eval/ci-20** → 点这次运行 → **Traces**，点开任意一道题，能看到这两层和每层的内容。

### 面试怎么说
> "OpenAI autologging puts every model call inside the question's trace — prompt, response, tokens, latency — with no change to the calling code."

---

## 第 5 步 · 思考模式值不值

### 在干什么
DeepSeek 新版模型默认会先"思考"（推理）再回答。用同样 20 道题比较"思考"和"不思考"两种模式。

### 背景
思考模式在很多任务上更准，但它要先写几千字的推理，才给出答案：**更慢、更贵**。值不值，不能凭感觉，要用评测说话。

### 用了什么技术，为什么
两种模式是两个"系统"：`llm-direct`（关掉思考）和 `llm-direct-thinking`（打开思考），在同一份 ci-20 上各跑一次（已经跑过，记录在 MLflow 里，这里直接读）。

| | 不思考 | 思考 |
|---|---|---|
| 前 1 名 | 15% | 10% |
| 前 10 名 | 30%（再跑一次是 40%） | 25% |
| 每道题用时 | 2 秒 | 43 秒 |
| 20 道题的输出 token | 8 千 | 21 万 |
| 20 道题的花费（高峰价） | 0.01 美元 | 0.25 美元 |

**20 道题上，思考模式没有更好**，但**慢 20 倍、贵 25 倍左右**。注意不思考模式在同样 20 道题上跑了两次，前 10 名一次 30%、一次 40%：只差两道题。20 道题的分数就是会这样跳，所以两种模式在这里其实分不出高下。所以 200 道题的评测先只用不思考模式。思考模式要不要在 200 道题上跑（约 1.2–2.5 美元），等需要时再定。

### 公司里什么情况怎么做
- **用评测做选择**：模型、模式、提示词，每个选择都在同一份题上比，记下质量、花费和速度三样
- **小题集只用来筛**：20 道题分不出谁更好时，要么上大题集，要么按花费和速度选

### 执行
读出 ci-20 上记录过的运行：

In [ ]:
run("uv run raredx eval runs --split ci-20")

### 结果怎么看
列表里能看到 `llm-direct-thinking` 和 `llm-direct`（还有第 3 步刚跑的那一次）。花费和用时在 MLflow 网页上每次运行的 **Metrics** 里：`usage/cost_usd_estimate`、`usage/seconds_per_case`。

### 面试怎么说
> "Reasoning mode cost about 25 times more and was 20 times slower on the smoke set with no measurable gain, so the baseline runs without it; the choice was made on the evaluation, not on reputation."

---

## 第 6 步 · 大考和看错题

### 在干什么
在 dev-200（200 道题）上的正式成绩，以及错在哪里。

### 背景
只有一个分数，不知道下一步该改什么。公司里每次评测完，都会翻一翻错题。

### 用了什么技术，为什么
200 道题已经用干净的代码跑过（约 0.11 美元），记录在 MLflow 里：

| | 前 1 名 | 前 3 名 | 前 5 名 | 前 10 名 |
|---|---|---|---|---|
| 病（OMIM 编号对） | 14.5% | 21.0% | 25.0% | **28.0%**（22.2–34.6） |
| 基因 | 17.5% | 26.0% | 30.0% | 34.5% |
| 对照：瞎猜 | 0.5% | 2.5% | 4.5% | 9.0% |

和瞎猜比，前 10 名是它的 3 倍。作为参考，已发表的研究里，各种系统在 Phenopacket 这类题上第一名就答对的平均约 22%。

> **同样的设置跑两次，分数会差一点。** 代码提交前跑的那次，前 1 名是 13.5%；提交后重跑是 14.5%，都在误差范围里。大模型即使把随机性调到最低，回答也不是每次都完全一样。所以只报干净代码跑出来的那次，并且总是带着误差范围。

### 公司里什么情况怎么做
- **错题分类**：先把错题按原因分几类，再决定改哪里。改提示词、加工具、换模型，解决的是不同的错
- **看几道具体的题**：数字之外，一定要读几道错题的原文

### 执行
从 MLflow 里取出这次 200 道题的回答，按错的原因分一分：

In [ ]:
from raredx.evals import scoring
from raredx.evals.harness import candidates_of

clean = next(r for r in harness.runs("dev-200") if r["name"] == "llm-direct" and not r["code"].endswith("-dirty"))
rows = []
for trace in harness.traces(clean["run_id"]):
    root = trace.data.spans[0]
    case = by_id[root.inputs["question"]["case_id"]]
    answer = candidates_of(root.outputs)
    disease_rank, gene_rank = scoring.ranks(answer, case.disease_id, case.gene)
    rows.append((case, answer, disease_rank, gene_rank))

right_gene_wrong_number = [r for r in rows if scoring.found(r[3], 10) and not scoring.found(r[2], 10)]
print(f"200 道题里，前 10 名有正确的病：{sum(scoring.found(r[2], 10) for r in rows)} 道")
print(f"基因对了、但 OMIM 编号不对：{len(right_gene_wrong_number)} 道，例如：\n")
for case, answer, _, gene_rank in right_gene_wrong_number[:3]:
    guess = answer[gene_rank - 1]
    print(f"  正确答案  {case.disease_id}  {case.gene:<7} {case.disease_label}")
    print(f"  模型回答  {scoring.normalize_omim(guess.disease_id)}  {guess.gene:<7} {guess.disease_name}\n")

### 结果怎么看
- **前 10 名有正确的病**：56 道（就是 28%）
- **基因对了、编号不对**：15 道。看例子会发现，很多其实"差不多对"：
  - DiGeorge 综合征（188400）和"22q11.2 缺失综合征"（192430）：临床上几乎是一回事，在 OMIM 里是两个条目
  - 枫糖尿症 Ib 型：病名一字不差，但模型给的是总条目的编号，正确的是 2023 年新建的子条目
  - UMOD 相关的肾病：模型用的是这个病的旧名字和旧编号
  
  这说明按编号判分偏严。基因分能补上一部分；以后可以加一条"同一类病算半对"的规则（判分规则改了，用第 08 章的"重新判分"就行，不用重跑）
- **Perrault 7 型**：两种模式都只列出了 1–5 型。第 7 型是 2024 年底才报道的，很可能在模型的训练资料之后。**这正是 agent 要解决的：遇到模型不知道的新知识，要会去查**

### 面试怎么说
> "The single-call baseline reaches 28% top-10 on the dev set against a 9% floor. Error analysis shows a share of misses are near-misses at the OMIM-entry level, and that the model misses diseases described after its training data — which is the case for giving the agent literature tools."

---

## 本章小结

| 步骤 | 技术 | 公司里的作用 |
|---|---|---|
| 1 密钥和接口 | `.env` + `SecretStr` + OpenAI 标准接口 | 密钥不外泄；换模型只改配置 |
| 2 提示词和格式 | 版本号 + JSON 模式 + pydantic + 补救一次 | 答卷能自动判分；格式错误有记录 |
| 3 先小考 | ci-20 | 一分钱试通整条路 |
| 4 记录每次调用 | `mlflow.openai.autolog()` | 看得到模型收到什么、答了什么、花了多少 |
| 5 思考模式 | 同题比较质量、花费、速度 | 用评测做选择 |
| 6 大考和错题 | dev-200 + 错题分类 | 第一个可信的数字，和下一步改什么 |

**一句话**：大模型自己猜，200 道题前 10 名答对 28%，是瞎猜的 3 倍；它不认识训练之后才报道的病，这正是接下来 agent 要用工具去补的。

## 面试题

1. **How do you integrate an LLM so you can switch providers?**
   > "Through the OpenAI-compatible API with the official SDK. DeepSeek, most hosted providers and self-hosted vLLM speak it, so the provider is two settings — base URL and model name."

2. **How do you get reliable structured output?**
   > "JSON mode, a pydantic schema on the reply, one repair attempt that shows the model its validation error, and a recorded miss if it still fails — never a silent drop."

3. **How do you track the cost of LLM calls?**
   > "Autologging records token usage on every call's span; each evaluation run sums them into input and output tokens, seconds per case and a cost estimate, so model choices are compared on quality, cost and latency together."

4. **Would you use the model's reasoning mode?**
   > "Only if the evaluation says so. On our smoke set it was about 25 times more expensive and 20 times slower with no measurable gain, so the baseline runs without it."

5. **What does your baseline tell you about building the agent?**
   > "It sets the bar — 28% top-10 against a 9% floor — and the misses show where tools should help: diseases newer than the model's training data, and getting the exact OMIM entry right."

**下一章 10 · 第一个工具**：按症状给所有罕见病排序。不用大模型的老方法，做成 agent 的第一个工具，也是第二个基线。